# TF-IDF
En este cuadernillo se aplica TF-IDF para ofrecer recomendaciones basadas en contenido.   
Para ello se utiliza como fuente de datos PLN.parquet.   
Fuentes:  
https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html  
http://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html

Para aplicar TF-IDF es necesario hacer un preprocesamiento de texto

In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet("../../data/03_model_ready/NLP.parquet")

Se convierte todo a minúsculas

In [3]:
df['overview'] = df['overview'].str.replace(r'[^a-zA-Z\s]', '', regex=True).str.lower() 

Se utiliza spacy para preprocesamiento

In [4]:
# hay que añadir uv add click
# es necesario descargar el modelo: uv run python -m spacy download en_core_web_sm
# carga modelo preentrenado.
import spacy 
nlp = spacy.load("en_core_web_sm") 

Se realiza conversión a minúsculas, se quitan stopwords, se reduce las palabras sa su lema.

In [5]:
def prepocesamiento(texto):
    doc = nlp(texto.lower()) #minuscula y objeto doc
    tokens = [
        token.lemma_
        for token in doc
            if not token.is_stop and token.lemma_.strip() != ""]
    return " ".join(tokens)

se aplica a cada texto la función anterior y se crea la columna texto-tfidf

In [ ]:
df["texto_TFIDF"] = df["join"].apply(prepocesamiento)

In [ ]:
df.head()

## TFIDF

Se importan los conjuntos train y test. El de validacion no es necesario

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
df[df['genres'] ==""].head() # no hay nulos porque los quite (cambie null-> "")!!

Veamos primero como funciona la librería

In [ ]:
#corpus de documentos 
corpus = df['texto_TFIDF'].tolist()

# objeto 
vectorizer = TfidfVectorizer()

# fit aprende el vocabulario completo y transform vectoriza cada documento
# se tiene una coordenada por cada palabra del vocabulario
X = vectorizer.fit_transform(corpus)

# muestra las palabras que forman el vocabulario
vectorizer.get_feature_names_out()

In [ ]:
X.toarray() #matriz de vecrores content(s)

In [ ]:
#dataframe de la matriz anterior
# df_tfidf_content = pd.DataFrame(
#     X.toarray(),
#     columns=vectorizer.get_feature_names_out(),
#     index=df['movieId']
# )
# df_tfidf_content


In [ ]:
# por falt de memoria 

# X ya es sparse (la salida de TfidfVectorizer)
# Crear un mapping de movieId a índice en X
train =  pd.read_parquet("../../data/03_model_ready/ratings_train.parquet")
movie_to_idx = {mid: i for i, mid in enumerate(df['movieId'])}

perfiles = []
for user_id, group in train.groupby('userId'):
    indices = [movie_to_idx[m] for m in group['movieId'] if m in movie_to_idx]
    if indices:
        vectores = X[indices]  # sigue siendo sparse, no consume memoria
        pesos = group.loc[group['movieId'].isin(movie_to_idx), 'rating'].values
        # Media ponderada: sum(vector_i * peso_i) / sum(pesos)
        perfil = np.array((vectores.T @ pesos) / pesos.sum()).flatten()
    else:
        perfil = np.zeros(len(corpus))
    perfiles.append({'userId': user_id, 'BasedProfile': perfil})

df_tfidf_BasedProfile = pd.DataFrame(perfiles)

In [ ]:
#dimension (numero películas, vocabulario)
print(X.shape)

In [ ]:
# cosine_similarity(X, X)

Para buscar la pelicula más parecida a una película dada, se debe comparar el vector de la pelicula i con todos los demás. Por ejemplo película 1 (índice 0)

In [ ]:
# IndicePel = 0
# similitudes = cosine_similarity(X[IndicePel],X)[0] #devulve una matriz fila
# # poner a 0 la similitud consigo misma
# similitudes[IndicePel] = 0
# similitudes


In [ ]:
# #sacar el maximo
# similitudes.max() #valor
# similitudes.argmax() # indice de la peli con arg max

In [ ]:
# df.iloc[similitudes.argmax()]

# TF-IDF 

Se realiza el método sobre el conjunto de datos ratings con partición.

In [ ]:
#conjunto TOTAL de calificaciones
dfUser = pd.read_parquet("../../data/02_processed/ratings_integrity.parquet")
dfUser = dfUser.sort_values(['userId', 'timestamp'])
dfUser

Conjuntos train y test

In [ ]:
train =  pd.read_parquet("../../data/03_model_ready/ratings_train.parquet")
test =  pd.read_parquet("../../data/03_model_ready/ratings_test.parquet")

Siguiendo la teoría se debe crear un perfil del usuario con las películas que haya valorado. 

Para ello se usa el conjunto train y se agregan los vectores de cada usuario.

In [ ]:
# datos = pd.merge(df_tfidf_content, train, on = 'movieId', how = 'right')

# generos = vectorizer.get_feature_names_out().tolist()


# # hay peliculas que no tienen genero asociado
# datos[generos] = datos[generos].fillna(0)

# #usando la media ponderada por pesos(ratings). usuario es DF
# def agregacion(usuario):
#     vectores= usuario[generos].values
#     pesos    = usuario['rating'].values
#     return np.average(vectores, axis = 0, weights=pesos) # 0 es para que sea por columna

# df_tfidf_BasedProfile = datos.groupby('userId').apply(agregacion).reset_index() 

# df_tfidf_BasedProfile.columns=['userId', 'BasedProfile']
# df_tfidf_BasedProfile

In [ ]:
# df.columns 

para un usuario concreto la pelicula recomendada se hace calculando el item que maximiza ( la similitud de el vectorBasedProfile(usuario), todos los vectores de peliculas)



In [ ]:
# userId = 40 #usuario objetivo
# #vector de usuario
# vector_BP = df_tfidf_BasedProfile[df_tfidf_BasedProfile['userId' ]==userId]['BasedProfile'].values[0]

# #calculo de similitud entre el vector de usuario y la matriz de vectores conteido. uso coseno
# similitudes = cosine_similarity([vector_BP], X)[0]
# similitudes

In [ ]:
# datosUserId =  pd.DataFrame({
#     'movieId' : df['movieId'],
#     'title' :df['title'],
#     'similitud' : similitudes
#     })
# datosUserId

In [ ]:
# se deben quitar las peliuclas ya vistas !! Estas peliuclas son las de train
# pelisVistas = train[train['userId']==userId]['movieId'].tolist()
# pelisVistas

Lista de peliculas que el usuario no ha visto que se pueden recomendar. Se ordenan por similitud y se muestran las 10 primeras

In [ ]:
# pelisRec = datosUserId[~datosUserId['movieId'].isin(pelisVistas)].sort_values('similitud',ascending =False).head(10)
# pelisRec

veamos ahora si aparece alguna en test

In [ ]:
# resultados = pd.merge(test[test['userId']==userId], pelisRec, on="movieId", how = 'right')
# resultados[~resultados['rating'].isna()]


In [ ]:
def evaluacion(userId, k=10, t=4):
    vector_BP = df_tfidf_BasedProfile[df_tfidf_BasedProfile['userId' ]==userId]['BasedProfile'].values[0]
    similitudes = cosine_similarity([vector_BP], X)[0]
    datosUserId =  pd.DataFrame({
    'movieId' : df['movieId'],
    'title' :df['title'],
    'similitud' : similitudes
    })
    pelisVistas = train[train['userId']==userId]['movieId'].tolist()
    pelisRec = datosUserId[~datosUserId['movieId'].isin(pelisVistas)].sort_values('similitud',ascending =False).head(k)
    datos = pd.merge(test[test['userId']==userId], pelisRec, on="movieId", how = 'right')
    interseccion = datos[~datos['rating'].isna()] #peliculas vistas y que han sido recomendadas

    nRelevantes = (interseccion['rating'] >= t).sum()
    nTest = (test[test["userId"]==userId]['rating'] >= t).sum() #pelis test relevantes
    precisionk = nRelevantes/k
    if nTest !=0:
        recalk = nRelevantes/nTest
    else:
        recalk = 0
    if recalk + precisionk != 0:
        F1k = 2*recalk*precisionk/(recalk + precisionk)
    else:
        F1k = 0
  
    resultados =  pd.DataFrame({
        'userId' : [userId],
        'precisionK': [precisionk],
        'recalK':[recalk],
        'F1K':[F1k]
        })
    return resultados
    
def resultadosUmbral(t, p=1):
    listaUs = dfUser['userId'].unique()
    evaluacionTFIDF = pd.DataFrame()
    for i in listaUs:
        evaluacionTFIDF = pd.concat([evaluacionTFIDF,evaluacion(i,t=t)], ignore_index=True)

    solucion = evaluacionTFIDF[['precisionK', 'recalK','F1K']].mean()*p 
    return solucion


In [ ]:
resultadosUmbral(3.5,p=100)

In [ ]:
def evaluacionCAMBIO(userId, k=10, t=4):
    vector_BP = df_tfidf_BasedProfile[df_tfidf_BasedProfile['userId' ]==userId]['BasedProfile'].values[0]
    similitudes = cosine_similarity([vector_BP], X)[0]
    datosUserId =  pd.DataFrame({
    'movieId' : df['movieId'],
    'title' :df['title'],
    'similitud' : similitudes
    })
    pelisVistas = train[train['userId']==userId]['movieId'].tolist()
    pelisRec = datosUserId[~datosUserId['movieId'].isin(pelisVistas)].sort_values('similitud',ascending =False).head(k)
    datos = pd.merge(test[test['userId']==userId], pelisRec, on="movieId", how = 'right')
    interseccion = datos[~datos['rating'].isna()] #peliculas vistas y que han sido recomendadas

    nRelevantes = (interseccion['rating'] >= t).sum()
    nTest = (test[test["userId"]==userId]['rating'] >= t).sum() #pelis test relevantes
    precisionk = nRelevantes/k
    if nTest !=0:
        recalk = nRelevantes/nTest
    else:
        recalk = 0
    if recalk + precisionk != 0:
        F1k = 2*recalk*precisionk/(recalk + precisionk)
    else:
        F1k = 0
  
    resultados =  pd.DataFrame({
        'userId' : [userId],
        'precisionK': [precisionk],
        'recalK':[recalk],
        'F1K':[F1k]
        })
    return resultados
    
def resultadosUmbralCambio(t, p=1):
    listaUs = dfUser['userId'].unique()
    evaluacionTFIDF = pd.DataFrame()
    for i in listaUs:
        evaluacionTFIDF = pd.concat([evaluacionTFIDF,evaluacionCAMBIO(i,t=t)], ignore_index=True)

    solucion = evaluacionTFIDF[['precisionK', 'recalK','F1K']].mean()*p 
    return solucion


In [ ]:
resultadosUmbralCambio(3.5,p=100)